In [1]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import h5py
from tqdm.auto import tqdm
from PIL import Image
import torchvision.transforms as transforms # Needed for image standardization

# --- Configuration for Image Processing ---
# Base directory containing the mid-frame folders (1st_10min_mid, etc.)
IMAGE_FRAME_BASE_DIR = "/home/poorna/Downloads/data_frame" # <-- UPDATE IF NECESSARY
# Standard size for the raw input image pixels (e.g., standard ViT/ResNet input)
TARGET_IMAGE_SIZE = (224, 224) 
# List of the 7 source directories (must be sorted correctly)
IMAGE_SOURCE_DIRS = [
    "1st_10min_mid", "2nd_10min_mid", "3rd_10min_mid", "4th_10min_mid", 
    "5th_10min_mid", "6th_10min_mid", "7th_10min_mid"
]
IMAGE_EXTENSION = ".jpg" 

# --- Define the standard transformations for image loading ---
# Convert to tensor and resize/crop. Normalization can be applied during training.
image_transform = transforms.Compose([
    transforms.Resize(TARGET_IMAGE_SIZE),
    transforms.ToTensor(), # Converts H x W x C (0-255) to C x H x W (0.0-1.0)
])

# ---------------------------------------------------------------------

# --- 1. EEG-IMAGE DATASET CLASS ---

class EEGImageDataset(Dataset):
    """
    Loads EEG data and aligns it with corresponding RAW image pixel data.
    """
    def __init__(self, eeg_dir, image_frame_base_dir, image_source_dirs, transform):
        
        self.image_frame_base_dir = image_frame_base_dir
        self.transform = transform

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = [] 

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        num_subjects = len(self.eeg_file_paths)
        
        if num_subjects == 0:
             raise ValueError("No EEG files were loaded.")
             
        num_unique_stimuli = total_samples // num_subjects
        print(f"Found {num_subjects} EEG files → Total samples: {total_samples}")
        print(f"Calculated number of unique stimuli (images): {num_unique_stimuli}")


        # -------------------------
        # 2. Map Image Files to Stimuli Order
        # -------------------------
        self.image_file_paths = []

        for folder_name in image_source_dirs:
            folder_path = os.path.join(image_frame_base_dir, folder_name)
            
            if not os.path.exists(folder_path):
                 raise FileNotFoundError(f"Image folder not found: {folder_path}. Check your path.")

            # Get all image files, and crucially, SORT them to match EEG order
            image_files_in_folder = sorted([
                os.path.join(folder_path, f) 
                for f in os.listdir(folder_path) 
                if f.endswith(IMAGE_EXTENSION)
            ])
            
            self.image_file_paths.extend(image_files_in_folder)

        if len(self.image_file_paths) != num_unique_stimuli:
             raise ValueError(
                 f"Data Mismatch: Found {len(self.image_file_paths)} image files but expected {num_unique_stimuli} unique stimuli. "
             )
        
        # Now, repeat the image paths for each subject
        self.image_paths_repeated = self.image_file_paths * num_subjects
        
        print(f"Total image paths aligned: {len(self.image_paths_repeated)}")


    def __len__(self):
        return len(self.index_map)

    def get_image_pixels(self, image_path):
        """
        Loads the image, applies the standard transform (resize, ToTensor), 
        and returns the C x H x W pixel tensor.
        """
        try:
            image = Image.open(image_path).convert("RGB")
            # Apply transformation (resize, ToTensor, etc.)
            pixel_tensor = self.transform(image)
            return pixel_tensor
            
        except Exception as e:
            print(f"Warning: Could not process image {image_path}: {e}. Returning zero tensor.")
            # Default C=3 (RGB), H=224, W=224
            return torch.zeros((3, TARGET_IMAGE_SIZE[0], TARGET_IMAGE_SIZE[1]), dtype=torch.float32)


    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        
        # 1. EEG Tensor
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        
        # 2. Raw Image Pixel Tensor
        image_path = self.image_paths_repeated[idx]
        image_pixel_tensor = self.get_image_pixels(image_path)
        
        # RETURN EEG and RAW IMAGE PIXELS (2 TENSORS)
        return eeg_tensor, image_pixel_tensor

# ---------------------------------------------------------------------

# --- 3. HDF5 GENERATION SCRIPT (EEG + RAW IMAGE PIXELS) ---

# --- Paths (Set by user) ---
HDF5_FILE = "/home/poorna/data/eeg_raw_image_final.h5" # <-- NEW OUTPUT FILE NAME
EEG_DIR = "/home/poorna/data/preprocessed_eeg"

# --- Dataset Initialization ---
print("\nInitializing dataset...")

dataset = EEGImageDataset(
    eeg_dir=EEG_DIR,
    image_frame_base_dir=IMAGE_FRAME_BASE_DIR,
    image_source_dirs=IMAGE_SOURCE_DIRS,
    transform=image_transform # Pass the transformation pipeline
)

loader = DataLoader(dataset, batch_size=1, shuffle=False)

# --- Get dynamic shapes from a sample ---
n_samples = len(dataset)

if n_samples == 0:
    print("Error: Dataset is empty.")
else:
    # Get 2 items from the dataset
    sample_eeg, sample_image_pixels = dataset[0]

    # --- Create HDF5 file ---
    print(f"\nCreating NEW HDF5 file at {HDF5_FILE}...")
    with h5py.File(HDF5_FILE, "w") as f:
        eeg_shape = (n_samples, *sample_eeg.shape)
        # The image shape is (N, C, H, W)
        image_pixel_shape = (n_samples, *sample_image_pixels.shape) 

        print(f"Allocating space for {n_samples} samples...")
        print(f"  - EEG shape: {eeg_shape} (N, 62, 400)")
        print(f"  - Image Pixel shape: {image_pixel_shape} (N, 3, {TARGET_IMAGE_SIZE[0]}, {TARGET_IMAGE_SIZE[1]})") 

        eeg_ds = f.create_dataset("eeg", shape=eeg_shape, dtype="float32")
        # Store as float32 since ToTensor converts pixels to 0.0-1.0 floats
        image_pixel_ds = f.create_dataset("image_pixels", shape=image_pixel_shape, dtype="float32") 

        # Iterate and save
        print("Writing data to HDF5 file...")
        try:
            # Unpack 2 tensors from the loader
            for idx, (eeg_tensor, image_pixel_tensor) in enumerate(tqdm(loader, desc="Saving to HDF5")):
                eeg_ds[idx] = eeg_tensor.squeeze(0).numpy()
                image_pixel_ds[idx] = image_pixel_tensor.squeeze(0).numpy()
            
            print(f"\nSuccessfully saved new dataset to {HDF5_FILE}")

        except Exception as e:
            print(f"\n--- ERROR during HDF5 writing at index {idx} ---")
            print(f"Error: {e}")


Initializing dataset...
Found 20 EEG files → Total samples: 28000
Calculated number of unique stimuli (images): 1400
Total image paths aligned: 28000

Creating NEW HDF5 file at /home/poorna/data/eeg_raw_image_final.h5...
Allocating space for 28000 samples...
  - EEG shape: (28000, 62, 400) (N, 62, 400)
  - Image Pixel shape: (28000, 3, 224, 224) (N, 3, 224, 224)
Writing data to HDF5 file...


Saving to HDF5:   0%|          | 0/28000 [00:00<?, ?it/s]


Successfully saved new dataset to /home/poorna/data/eeg_raw_image_final.h5


In [4]:
import h5py
import numpy as np

# --- Configuration (Match paths from your generation script) ---
HDF5_FILE = "/home/poorna/data/eeg_raw_image_final.h5"
NUM_SAMPLES_TO_CHECK = 5

print(f"--- Inspecting first {NUM_SAMPLES_TO_CHECK} entries in {HDF5_FILE} ---")

try:
    with h5py.File(HDF5_FILE, "r") as f:
        print("\nDatasets available:", list(f.keys()))

        if "eeg" not in f or "image_pixels" not in f:
            print("Error: Required datasets ('eeg', 'image_pixels') not found in the HDF5 file.")
            exit()

        for i in range(NUM_SAMPLES_TO_CHECK):
            eeg_data = f["eeg"][i]
            image_pixels = f["image_pixels"][i]

            print(f"\n--- Sample {i} ---")
            print(f"EEG Data:")
            print(f"  - Shape: {eeg_data.shape} (Channels=62, Timesteps=400)")
            print(f"  - Dtype: {eeg_data.dtype}")
            print(f"Image Pixels Data:")
            print(f"  - Shape: {image_pixels.shape} (Channels=3, Height=224, Width=224)")
            print(f"  - Dtype: {image_pixels.dtype}")
            
except FileNotFoundError:
    print(f"\nFATAL ERROR: HDF5 file not found at {HDF5_FILE}. Please ensure the generation script ran successfully.")
except Exception as e:
    print(f"\nAn unexpected error occurred while reading the file: {e}")

--- Inspecting first 5 entries in /home/poorna/data/eeg_raw_image_final.h5 ---

Datasets available: ['eeg', 'image_pixels']

--- Sample 0 ---
EEG Data:
  - Shape: (62, 400) (Channels=62, Timesteps=400)
  - Dtype: float32
Image Pixels Data:
  - Shape: (3, 224, 224) (Channels=3, Height=224, Width=224)
  - Dtype: float32

--- Sample 1 ---
EEG Data:
  - Shape: (62, 400) (Channels=62, Timesteps=400)
  - Dtype: float32
Image Pixels Data:
  - Shape: (3, 224, 224) (Channels=3, Height=224, Width=224)
  - Dtype: float32

--- Sample 2 ---
EEG Data:
  - Shape: (62, 400) (Channels=62, Timesteps=400)
  - Dtype: float32
Image Pixels Data:
  - Shape: (3, 224, 224) (Channels=3, Height=224, Width=224)
  - Dtype: float32

--- Sample 3 ---
EEG Data:
  - Shape: (62, 400) (Channels=62, Timesteps=400)
  - Dtype: float32
Image Pixels Data:
  - Shape: (3, 224, 224) (Channels=3, Height=224, Width=224)
  - Dtype: float32

--- Sample 4 ---
EEG Data:
  - Shape: (62, 400) (Channels=62, Timesteps=400)
  - Dtype: flo